In [8]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
from vis_statistics import plot_cluster_statistics, plot_clusters_comparison
from vis_lc import get_lc_path, plot_cluster_lcs
from vis_periodogram import plot_cluster_periodograms, plot_cluster_periodograms_specific, plot_lsp_comparison

## Load master table

In [10]:
from astropy.table import Table

tbl = Table.read('master_table.fits')

scalar_cols = [name for name in tbl.colnames if len(tbl[name].shape) <= 1]
multi_cols  = [name for name in tbl.colnames if len(tbl[name].shape) >  1]

master_table = tbl[scalar_cols].to_pandas()

for col in multi_cols:
    master_table[col] = list(tbl[col])

print(master_table.shape)
master_table.head()

(868, 26)


,name,rms,std,MAD,sigmaG,skewness,von_neumann_ratio,J_Stetson,max_power,freq_at_max_power,...,SumLSP_7_4_Day_Power,SumLSP_4_1_Day_Power,SumLSP_1_p5_Day_Power,Entropy,sectors,cadence,n_sectors,LOC,AGE,FullPeriodogram
0,b'BMG32',0.061409,0.000668,0.000434,0.000639,0.397087,0.000783,0.935988,0.046111,0.36,...,0.206012,0.528081,0.173964,6.981934,[0],30.0,1,b'LMC',7.9,"[0.026436947727934403, 0.02746316684230981, 0...."
1,b'BMG32',0.161769,0.001006,0.000719,0.001065,-3.085452,0.001452,1.434224,0.174483,0.32,...,0.170952,1.841990,0.271482,6.934397,[1],30.0,1,b'LMC',7.9,"[0.002725964423310637, 0.007146544857656363, 0..."
2,b'BMG32',0.112893,0.001003,0.000626,0.000927,17.266732,0.001003,1.377425,0.174913,0.19,...,0.702348,1.005133,0.103512,7.047517,[2],30.0,1,b'LMC',7.9,"[0.05205991154852454, 0.07989218713552883, 0.0..."
3,b'BMG32',0.119027,0.001056,0.000722,0.001067,-2.638127,0.001146,1.451241,0.112790,0.37,...,0.451855,1.091927,0.113481,7.156956,[3],30.0,1,b'LMC',7.9,"[0.0703332776487653, 0.05668259486342891, 0.03..."
4,b'BMG32',0.061409,0.000850,0.000546,0.000811,-1.767441,0.000560,1.179916,0.068967,0.37,...,0.100536,0.647467,0.117497,7.651595,"[0, 1]",30.0,2,b'LMC',7.9,"[0.006778285831005086, 0.013017152060444029, 0..."


In [11]:
print(master_table.columns)

Index(['name', 'rms', 'std', 'MAD', 'sigmaG', 'skewness', 'von_neumann_ratio',
       'J_Stetson', 'max_power', 'freq_at_max_power', 'n_peaks',
       'ratio_of_power_at_high_v_low_freq', 'FAP', 'max_autocorrelation',
       'time_of_max_autocorrelation', 'SumLSP_10_7_Day_Power',
       'SumLSP_7_4_Day_Power', 'SumLSP_4_1_Day_Power', 'SumLSP_1_p5_Day_Power',
       'Entropy', 'sectors', 'cadence', 'n_sectors', 'LOC', 'AGE',
       'FullPeriodogram'],
      dtype='object')


In [12]:
new_table = master_table[['name', 'sectors', 'cadence', 'n_sectors', 'LOC', 'AGE']].copy()

In [13]:
new_table

,name,sectors,cadence,n_sectors,LOC,AGE
0,b'BMG32',[0],30.0,1,b'LMC',7.90
1,b'BMG32',[1],30.0,1,b'LMC',7.90
2,b'BMG32',[2],30.0,1,b'LMC',7.90
3,b'BMG32',[3],30.0,1,b'LMC',7.90
4,b'BMG32',"[0, 1]",30.0,2,b'LMC',7.90
...,...,...,...,...,...,...
863,b'SAURER 6',"[2, 3]",10.0,2,b'MW',9.20
864,b'SAURER 6',"[1, 2, 3]",10.0,3,b'MW',9.20
865,b'VDBERGH-HAGEN 92',[0],30.0,1,b'MW',8.91
866,b'VDBERGH-HAGEN 92',[1],30.0,1,b'MW',8.91


## Build updated table

In [14]:
from extract_LC_LSP import add_lightcurves, add_lsps
from invariant_summary_stats import add_variability_metrics
from invariant_LSP import add_invariant_LSP_stats

add_lightcurves(new_table)
add_lsps(new_table, P_min=0.1, P_max=10.0, alpha=5, fap_level=0.01)

[add_lightcurves] processing 868 rows across 55 clusters  |  cadence=30.0 min
[add_lightcurves] done  |  ok=868  missing=0  failed=0
[add_lsps] processing 868 rows  |  P=[0.1, 10.0] days  alpha=5  FAP=0.01  T_effective=True
[add_lsps] done  |  ok=868  skipped=0  failed=0


,name,sectors,cadence,n_sectors,LOC,AGE,LC_t,LC_x,LC_err_x,LSP_freq,LSP_power,LSP_FAP
0,b'BMG32',[0],30.0,1,b'LMC',7.90,"[1382.0534040342682, 1382.0742371425345, 1382....","[1.0000975450628886, 0.9998390715786509, 0.998...","[0.0006122919607540418, 0.0006073373966713103,...","[0.1, 0.10839170102580067, 0.11678340205160134...","[0.03138590632708135, 0.040288270937488116, 0....",0.025225
1,b'BMG32',[1],30.0,1,b'LMC',7.90,"[1410.9283805553364, 1410.9492137124166, 1411....","[0.9959518989804473, 0.9967784422446525, 0.998...","[0.0005083746925240335, 0.0005079584784039902,...","[0.1, 0.1090738263391421, 0.1181476526782842, ...","[0.000560826641981916, 0.0031543272050220434, ...",0.026706
2,b'BMG32',[2],30.0,1,b'LMC',7.90,"[1569.447644895273, 1569.468478380938, 1569.48...","[1.0013508768642108, 0.9995276507017979, 0.998...","[0.001107691177440487, 0.0011245434875321722, ...","[0.1, 0.1079403652130463, 0.1158807304260926, ...","[0.031273467477461146, 0.058572832576034586, 0...",0.023969
3,b'BMG32',[3],30.0,1,b'LMC',7.90,"[1653.948830404778, 1653.969663983743, 1653.99...","[1.000487143665834, 1.000670471719181, 1.00029...","[0.0004794438707843265, 0.0004792852712212794,...","[0.1, 0.10704321537201596, 0.1140864307440319,...","[0.08562990948547124, 0.09200512740769964, 0.0...",0.021693
4,b'BMG32',"[0, 1]",30.0,2,b'LMC',7.90,"[1382.0534040342682, 1382.0742371425345, 1382....","[1.0000975450628886, 0.9998390715786509, 0.998...","[0.0006122919607540418, 0.0006073373966713103,...","[0.1, 0.10435972165094515, 0.10871944330189029...","[0.0031286018190834317, 0.003848817359201781, ...",0.013784
...,...,...,...,...,...,...,...,...,...,...,...,...
863,b'SAURER 6',"[2, 3]",10.0,2,b'MW',9.20,"[2769.9167884451695, 2769.937622284224, 2769.9...","[0.9998862677333421, 0.999875633701568, 0.9997...","[4.274047492678074e-05, 4.273635535963352e-05,...","[0.1, 0.10426984111967216, 0.10853968223934432...","[0.0028019042016262643, 0.003643753164533522, ...",0.013278
864,b'SAURER 6',"[1, 2, 3]",10.0,3,b'MW',9.20,"[2420.0069345704387, 2420.027767841269, 2420.0...","[1.000409940549219, 0.9999722212505378, 1.0001...","[5.2359810627277134e-05, 5.2451677502881545e-0...","[0.1, 0.10276898072228992, 0.10553796144457982...","[0.0019369448212386287, 0.00294436948758637, 0...",0.009036
865,b'VDBERGH-HAGEN 92',[0],30.0,1,b'MW',8.91,"[1543.784412772034, 1543.80524656084, 1543.826...","[1.0003120573494295, 1.0006849320573654, 1.000...","[3.471778038086393e-05, 3.448285757645053e-05,...","[0.1, 0.1086174944495052, 0.1172349888990104, ...","[0.10587165306612822, 0.11418969050652931, 0.0...",0.025370
866,b'VDBERGH-HAGEN 92',[1],30.0,1,b'MW',8.91,"[1569.451349827374, 1569.472183108963, 1569.49...","[0.9999328665576012, 1.0000060038037917, 1.000...","[3.9436533405072194e-05, 4.0246802399471696e-0...","[0.1, 0.10794052408189338, 0.11588104816378675...","[0.10567532534874734, 0.1424068514684566, 0.12...",0.023970


In [15]:
add_variability_metrics(new_table)
add_invariant_LSP_stats(new_table)

[add_variability_metrics] processing 868 rows
[add_variability_metrics] done  |  ok=868  skipped=0  failed=0
[add_invariant_LSP_stats] processing 868 rows
[add_invariant_LSP_stats] done  |  ok=868  skipped=0  failed=0


,name,sectors,cadence,n_sectors,LOC,AGE,LC_t,LC_x,LC_err_x,LSP_freq,...,intrinsic_rms,intrinsic_mad,n_bins_used,LSP_max_power,LSP_freq_at_max_power,LSP_period_at_max_power,LSP_SumPow_10_7,LSP_SumPow_7_4,LSP_SumPow_4_1,LSP_SumPow_1_p5
0,b'BMG32',[0],30.0,1,b'LMC',7.90,"[1382.0534040342682, 1382.0742371425345, 1382....","[1.0000975450628886, 0.9998390715786509, 0.998...","[0.0006122919607540418, 0.0006073373966713103,...","[0.1, 0.10839170102580067, 0.11678340205160134...",...,0.000377,0.000332,1077.0,0.080666,0.360143,2.776677,0.185001,0.419068,1.005505,0.269574
1,b'BMG32',[1],30.0,1,b'LMC',7.90,"[1410.9283805553364, 1410.9492137124166, 1411....","[0.9959518989804473, 0.9967784422446525, 0.998...","[0.0005083746925240335, 0.0005079584784039902,...","[0.1, 0.1090738263391421, 0.1181476526782842, ...",...,0.000765,0.000830,1027.0,0.240224,0.317772,3.146912,0.047830,0.298017,2.624035,0.353090
2,b'BMG32',[2],30.0,1,b'LMC',7.90,"[1569.447644895273, 1569.468478380938, 1569.48...","[1.0013508768642108, 0.9995276507017979, 0.998...","[0.001107691177440487, 0.0011245434875321722, ...","[0.1, 0.1079403652130463, 0.1158807304260926, ...",...,0.000707,0.000625,1150.0,0.237476,0.187344,5.337774,0.444140,1.160263,1.625045,0.133833
3,b'BMG32',[3],30.0,1,b'LMC',7.90,"[1653.948830404778, 1653.969663983743, 1653.99...","[1.000487143665834, 1.000670471719181, 1.00029...","[0.0004794438707843265, 0.0004792852712212794,...","[0.1, 0.10704321537201596, 0.1140864307440319,...",...,0.000784,0.000799,1283.0,0.152587,0.367642,2.720036,0.439434,0.934364,1.873543,0.169371
4,b'BMG32',"[0, 1]",30.0,2,b'LMC',7.90,"[1382.0534040342682, 1382.0742371425345, 1382....","[1.0000975450628886, 0.9998390715786509, 0.998...","[0.0006122919607540418, 0.0006073373966713103,...","[0.1, 0.10435972165094515, 0.10871944330189029...",...,0.000599,0.000565,2104.0,0.108654,0.374662,2.669069,0.077800,0.356884,2.211132,0.344050
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
863,b'SAURER 6',"[2, 3]",10.0,2,b'MW',9.20,"[2769.9167884451695, 2769.937622284224, 2769.9...","[0.9998862677333421, 0.999875633701568, 0.9997...","[4.274047492678074e-05, 4.273635535963352e-05,...","[0.1, 0.10426984111967216, 0.10853968223934432...",...,0.000203,0.000158,2189.0,0.154791,0.642270,1.556978,0.039292,0.130958,1.229535,1.306176
864,b'SAURER 6',"[1, 2, 3]",10.0,3,b'MW',9.20,"[2420.0069345704387, 2420.027767841269, 2420.0...","[1.000409940549219, 0.9999722212505378, 1.0001...","[5.2359810627277134e-05, 5.2451677502881545e-0...","[0.1, 0.10276898072228992, 0.10553796144457982...",...,0.000180,0.000146,3405.0,0.076556,0.642720,1.555887,0.094592,0.236637,1.091622,1.165372
865,b'VDBERGH-HAGEN 92',[0],30.0,1,b'MW',8.91,"[1543.784412772034, 1543.80524656084, 1543.826...","[1.0003120573494295, 1.0006849320573654, 1.000...","[3.471778038086393e-05, 3.448285757645053e-05,...","[0.1, 0.1086174944495052, 0.1172349888990104, ...",...,0.000120,0.000099,1084.0,0.131199,0.246497,4.056838,0.390390,0.676576,1.260146,0.390746
866,b'VDBERGH-HAGEN 92',[1],30.0,1,b'MW',8.91,"[1569.451349827374, 1569.472183108963, 1569.49...","[0.9999328665576012, 1.0000060038037917, 1.000...","[3.9436533405072194e-05, 4.0246802399471696e-0...","[0.1, 0.10794052408189338, 0.11588104816378675...",...,0.000187,0.000122,1150.0,0.142407,0.107941,9.264361,0.478879,0.554786,0.793889,0.320582


## Save

In [17]:
# from astropy.table import Table
# import numpy as np

# # LC arrays are large intermediates — drop before saving; regenerate with add_lightcurves()
# save_table = new_table.drop(columns=['LC_t', 'LC_x', 'LC_err_x'])

# # Separate scalar columns from per-row array columns
# def _is_array_col(series):
#     first = next((v for v in series if v is not None), None)
#     return isinstance(first, (np.ndarray, list))

# scalar_cols = [c for c in save_table.columns if not _is_array_col(save_table[c])]
# array_cols  = [c for c in save_table.columns if     _is_array_col(save_table[c])]

# tbl = Table.from_pandas(save_table[scalar_cols])
# for col in array_cols:
#     tbl[col] = [v if v is not None else np.array([]) for v in save_table[col]]

# tbl.write('new_master_table.fits', overwrite=True)
# print(f"Saved {len(tbl)} rows  |  columns: {tbl.colnames}")